Анализ качества предсказаний сигналов (на примере физики частиц, i.e., HIGGS) 
при использовании различных аугментаций, построенных на 

(а) hand-crafted symmetries, 

(b) automatically extracted symmetries (e.g., AtlasD) + добавление фич из TDA 

(интересно посмотреть, как они меняются при действии групповых свойств из automatically extracted symmetries)

Ниже — сквозной (end-to-end) пайплайн для задачи анализа качества предсказаний сигналов в HIGGS-задаче с фокусом на:

аугментации на основе

(a) hand-crafted симметрий

(b) автоматически извлечённых симметрий (типа Atlas-style / learned equivariances)

добавлении TDA-фич (Topological Data Analysis)

и исследовании того, как TDA-фичи “ведут себя” под действием групповых преобразований

Я буду описывать это как исследовательский ML-пайплайн (не production), с акцентом на эксперименты и измерения.

## 1. Постановка задачи

Берём стандартный датасет типа HIGGS (UCI / Kaggle):

* вход: 28 признаков (kinematic variables)
* задача: бинарная классификация
    -   signal (Higgs boson event)
    -   background (QCD processes)

Цель пайплайна:

улучшить качество классификации и устойчивость модели за счёт:
* симметрийных аугментаций
* learned symmetries
* TDA-фич
* анализа инвариантности/эквивариантности

# 2. Общая архитектура эксперимента

Мы строим 4 параллельных ветки:

* Baseline
    -    обычная модель без аугментаций
    -   без TDA
* Hand-crafted symmetry augmentation
    -   физически мотивированные преобразования
* Learned symmetry augmentation (Atlas-like)
    -   симметрии извлекаются из данных
* TDA-enhanced versions
    -   добавление топологических признаков
    -   анализ их стабильности под группой

## 3. Данные и предобработка
#### 3.1 Очистка
* нормализация признаков (z-score)
* обработка выбросов (clipping или robust scaler)
#### 3.2 Разбиение
* train / val / test (например 70/15/15)
* важно: фиксировать seed для сравнимости симметрий

## 4. Базовая модель

Используем несколько вариантов:

* MLP (baseline)
* Gradient Boosted Trees (XGBoost / LightGBM)
* optional: transformer/tabular model

Но для симметрий и TDA лучше:

* MLP или equivariant network (если есть структура)

## 5. (a) Hand-crafted symmetry augmentations
#### 5.1 Физически мотивированные симметрии в HIGGS

Примеры:

(1) rotational invariance (частично): повороты в transverse plane
```φ → φ + Δφ```

(2) permutation symmetries: перестановки ```jets (jet1 ↔ jet2)```

(3) reflection symmetry: ```φ → -φ``` (если допустимо в feature space)

(4) energy scaling invariance (частично): масштабирование pT с нормировкой

#### 5.2 Реализация augmentation pipeline
```
def hand_crafted_augment(x):
    if random() < p:
        x = rotate_phi(x, angle=random_uniform(0, 2π))
    if random() < p:
        x = permute_jets(x)
    if random() < p:
        x = reflect_phi(x)
    return x
```

#### 5.3 Важный момент
Не все фичи HIGGS физически симметричны → нужно:
* mask-based augmentation (разные правила для subsets of features)
* sanity checks (не ломаем label invariance)

## 6. (b) Automatically extracted symmetries (Atlas-style)

Здесь идея: вместо вручную заданных групп → учим скрытую группу преобразований

### 6.1 Подходы
#### Вариант A: Group discovery via neural equivariance

learn transformation gθ such that: ```f(x)≈f(gθ(x))```

оптимизация: минимизация prediction variance under transformations или contrastive loss
#### Вариант B: AtlasD-like idea (data-driven atlas of orbits)
* строим latent space (VAE / autoencoder)
* ищем:
    -   кластеры орбит (symmetry orbits)
    -   локальные преобразования между точками

#### Вариант C: Contrastive symmetry learning

Loss: $L=∣∣f(x)−f(T(x))∣∣^2$, 

где T — learned augmentation network

### 6.2 Выход системы
Мы получаем:
* learned transformation group Ĝ
* вероятностные аугментации
* локальные симметрии (state-dependent)

## 7. TDA слой (Topological Data Analysis)

Теперь добавляем топологические признаки.

### 7.1 Что извлекаем

Для каждого события (или batch embedding):
* persistence diagrams (H0, H1)
* Betti curves
* persistence images

### 7.2 Как строим point cloud

Варианты:

(A) feature space TDA: берём raw features → point cloud

(B) latent space TDA (лучше)
* encoder(x) → z ∈ R^d
* строим topology в latent space

### 7.3 Pipeline
```
x → encoder → latent z → Vietoris-Rips complex → persistence diagram → vectorization
```

### 7.4 Vectorization
* persistence images
* landscape functions
* entropy of persistence

## 8. Комбинированная модель

Мы объединяем:
* Input branch 1:
    -   raw features
* Input branch 2:
    -   TDA features
* Fusion:
    -   concat → MLP / attention layer

## 9. Ключевой эксперимент: влияние симметрий на TDA (Это центральная часть запроса).

### 9.1 Что проверяем

Для каждой группы:
* G (hand-crafted)
* Ĝ (learned)

Сравниваем:

(1) Stability of persistence diagrams --- $d_{Wasserstein​}(PD(x),PD(g(x)))$

(2) Topological invariance score --- насколько invariant TDA features under group actions

(3) Orbit collapse / expansion --- сжимается ли manifold при augmentation

### 9.2 Метрики
1. Stability score --- mean Wasserstein distance under group action
2. Equivariance error: $∣∣ϕ(gx)−g′ϕ(x)∣∣$
3. TDA variance ratio --- variance explained by group transformations


## 10. Training setup
### 10.1 Сценарии

Мы обучаем 6 моделей:
1. baseline
2. * hand-crafted augmentation
3. * learned symmetry augmentation
4. * TDA only
5. * hand-crafted + TDA
6. * learned + TDA

### 10.2 Loss functions
* BCE loss (classification)
    -   consistency loss (symmetry)
    -   optional contrastive loss (learned symmetries)

## 11. Evaluation
### 11.1 ML metrics
* AUC ROC
* accuracy
* F1 score
* calibration error
### 11.2 Robustness metrics
* performance under transformations
* adversarial augmentation tests
* OOD generalization
### 11.3 Topological metrics
* persistence stability
* Betti curve distance
* topological signature drift

##   12. Visualization

Очень важно:

(1) embedding space
* UMAP / t-SNE

(2) orbit structure
* trajectories x → g(x)

(3) persistence diagrams
* before/after augmentation

(4) symmetry-induced collapse
* manifold contraction

## 13. Научный смысл результата

фактически проверяется 3 гипотезы:

* (H1): Hand-crafted symmetries улучшают generalization, но ограничены.
* (H2): Learned symmetries находят более “реальные” invariances данных.
* (H3): TDA чувствителен к symmetry structure и может:
    -   либо стать invariant feature extractor
    -   либо “детектировать” нарушение симметрий

## 14. Потенциально интересные наблюдения
Обычно в таких пайплайнах появляется:
* TDA-фичи становятся более стабильными, если learned symmetries корректны
* hand-crafted symmetries могут “ломать” topology (если физически неточны)
* learned symmetries часто дают более “smooth” persistence landscapes
* TDA может выступать как детектор неправильных augmentations

## 15. Расширения (если углублять)
* Equivariant neural networks (E(3)-like, even if approximate)
* diffusion models для sampling symmetries
* group-conditional VAEs
* persistent homology regularization in loss
* geometric deep learning on learned manifolds

```mermaid
flowchart TD
A["Dataset"]
B["Preprocessing"]
C["Model"]
D["Evaluation"]

A --> B
B --> C
C --> D
```

Если хочешь, я могу дальше сделать одно из следующих:

* реальный PyTorch-псевдокод всей системы
* схему архитектуры (как research diagram)
* математическую формализацию learned symmetry discovery
* или дизайн эксперимента “как для NeurIPS paper”

схему архитектуры (как research diagram)
# 🧠 Общая архитектура пайплайна (HIGGS + Symmetry + TDA)
исследовательская архитектурная схема в формате, близком к NeurIPS-style system diagram.

```mermaid
flowchart TB

%% ===================== INPUT =====================
A[HIGGS Dataset\n#124;raw kinematic features#124;] --> B1[Preprocessing\n- normalization\n- feature masking]

%% ===================== PARALLEL BRANCHES =====================

B1 --> C1[Baseline Branch]
B1 --> C2[Hand-crafted Symmetry Augmentation]
B1 --> C3[Learned Symmetry Module #124;Atlas-style#124;]
B1 --> C4[TDA Feature Extraction]

%% ===================== HAND-CRAFTED =====================
C2 --> D2[Group Actions G\n- rotations\n- jet permutations\n- reflections]
D2 --> E2[Augmented Dataset X_g]

%% ===================== LEARNED SYMMETRY =====================
C3 --> D3[Symmetry Discovery Model]
D3 --> D3a["Equivariance Loss\nf#124;x#124; ~ f#124;g_theta#124;x#124;#124;"]
D3 --> D3b["Latent Group G_hat"]
D3 --> E3["Learned Augmentations X_g_hat"]

%% ===================== TDA =====================
C4 --> D4[Encoder #124;optional#124;]
D4 --> D5[Point Cloud / Latent Space]
D5 --> D6[Vietoris–Rips Complex]
D6 --> D7[Persistent Homology]
D7 --> D8[TDA Vectorization\n#124;persistence images, betti curves#124;]

%% ===================== MODEL BRANCHES =====================

C1 --> M1["Classifier f_theta"]
E2 --> M2["Classifier f_theta + Hand-crafted Aug"]
E3 --> M3["Classifier f_theta + Learned Symmetry"]
D8 --> M4[TDA-only Model]

%% ===================== FUSION MODEL =====================

C1 --> F0
E2 --> F0
E3 --> F0
D8 --> F0[Multimodal Fusion Model]

F0 --> F1[Concat / Attention Fusion]
F1 --> F2[Final Classifier]

%% ===================== OUTPUT =====================

M1 --> O[Evaluation]
M2 --> O
M3 --> O
M4 --> O
F2 --> O

O --> O1[AUC / Accuracy]
O --> O2[Robustness under group transforms]
O --> O3[TDA stability metrics]
O --> O4[Equivariance error]

%% ===================== ANALYSIS LAYER =====================

O --> A1[Symmetry-TDA Interaction Analysis]
A1 --> A2[Wasserstein distance between persistence diagrams]
A1 --> A3[Orbit stability / manifold collapse]
A1 --> A4[Invariant feature sensitivity]
```

## 🧩 Архитектура по блокам (интерпретация)
1. Input Layer
* HIGGS kinematic features
* preprocessing:
    - normalization
    - feature masking (jets / leptons / missing ET separation)

    2. Three symmetry pathways
**A. Hand-crafted symmetry branch**

Физически заданные группы:
* rotations in transverse plane
* jet permutations
* reflections

👉 output: augmented dataset $X_g$

**B. Learned symmetry branch (Atlas-style)**

Ключевая идея: модель сама находит группу преобразований

Компоненты:
* symmetry discovery network
* equivariance constraint: f(x)≈f(g_θ(x))
latent group $\hat G$

👉 output: learned augmented samples $X_{\hat g}$

**C. TDA branch**

Pipeline:

1. encoder (optional)
2. latent point cloud
3. Vietoris–Rips complex
4. persistent homology
5. vectorization:
    - persistence images
    - Betti curves
    - landscapes

👉 output: topological feature vector
	​
3. Model layer

Четыре режима:

1. baseline: raw features only
2. hand-crafted augmented: invariance via known symmetries
3. learned symmetry model: invariance via discovered group
4. TDA-only model: topology-driven classifier

4. Fusion model (ключевая часть)

Multimodal fusion:

raw features
+ symmetry-augmented views
+ learned symmetry views
+ TDA embeddings

→ attention / concat → classifier

5. Evaluation layer
ML metrics
* AUC ROC
* accuracy
* calibration
Physics robustness metrics
* invariance under transformations
Topological metrics
* stability of persistence diagrams
* Betti curve divergence
Symmetry diagnostics
* equivariance error
* orbit collapse/expansion

6. Analysis layer (самое исследовательское)

Здесь проверяется твоя основная научная гипотеза:

🔥 Interaction: symmetry × topology

Ты измеряешь:
* как group actions меняют topology
* стабильность persistent homology под:
    - hand-crafted G
    - learned Ĝ

Метрики:
* Wasserstein distance between persistence diagrams: $d_W (D(X),D(gX))$
* orbit stability score
* topology variance ratio

🧠 Как это выглядит концептуально (одной фразой)
Ты строишь систему, которая одновременно:
* учит симметрии данных
* навязывает физические симметрии
* и проверяет их через топологическую стабильность представлений


Если хочешь дальше, я могу:
* превратить это в NeurIPS-ready Figure caption + diagram description
* или нарисовать вторую схему: только mathematics (group theory + TDA coupling)
* или разложить loss functions для всех трёх веток

дизайн эксперимента “как для NeurIPS paper”. Ниже — дизайн эксперимента в стиле NeurIPS paper: с чёткими гипотезами, абляциями, протоколами, метриками и статистической проверкой. Я оформлю это так, как обычно ожидается в секции Experiments + Experimental Setup + Ablations.

# 🧪 Experimental Design (NeurIPS-style)
## 1. Goal of Experiments

Мы изучаем влияние:

1. hand-crafted symmetries
2. learned symmetries (Atlas-style discovery)
3. TDA-based representations

на:

* качество классификации HIGGS
* robustness к group transformations
* топологическую стабильность латентных представлений

## 2. Hypotheses
🔷 H1 — Physics priors help, but are incomplete

Hand-crafted symmetries улучшают generalization, но не покрывают все invariances.

🔷 H2 — Learned symmetries outperform manual ones

Data-driven group discovery выявляет более “true” invariances данных.

🔷 H3 — TDA encodes symmetry-sensitive structure

Persistent homology:
* инвариантна к правильным симметриям
* чувствительна к “нефизическим” augmentations

🔷 H4 — Symmetry + TDA synergy

Комбинация symmetry learning + TDA дает **лучший robustness–accuracy trade-off**

## 3. Dataset
Primary dataset
* HIGGS (UCI / Kaggle)
* binary classification:
    - signal (Higgs boson)
    - background
Split protocol
* Train: 70%
* Validation: 15%
* Test: 15%
Seeds
* 5 random seeds
* report mean ± std

## 4. Experimental Conditions

We evaluate 6 main models:

| ID | Model                        |
| -- | ---------------------------- |
| M0 | Baseline MLP                 |
| M1 | + Hand-crafted augmentation  |
| M2 | + Learned symmetries (Atlas) |
| M3 | + TDA features only          |
| M4 | Hand-crafted + TDA           |
| M5 | Learned symmetries + TDA     |


## 5. Symmetry Construction

### 5.1 Hand-crafted group G

Defined transformations:
* φ-rotations (transverse plane)
* jet permutations (j1 ↔ j2)
* sign flips in selected subspaces

Formal: $G = {g_1, g_2​,...,g_k}$

### 5.2 Learned group Ĝ
Model:
* symmetry discovery network $g_θ$​
* trained via equivariance loss:
$L_{sym} = E_{x,g}∥f(x)−f(g_θ(x))∥^2$

Output:
* stochastic transformation sampler
* latent group orbits

## 6. TDA Pipeline
### 6.1 Construction
Two variants:
(A) Raw-space TDA: $x ∈ R^{28}$
(B) Latent-space TDA (preferred): $z=encoder(x)$

### 6.2 Persistent Homology
* Vietoris–Rips complex
* compute:
    - H0 (connected components)
    - H1 (loops)

### 6.3 Vectorization
* persistence images (PI)
* Betti curves
* persistence entropy

## 7. Training Protocol
### 7.1 Optimizer
* Adam 
* lr = 1e-3 
* cosine decay

### 7.2 Batch size
* 256

### 7.3 Loss functions
**Classification loss**: $L_{cls}​ = BCE(y,\hat y)$

**Symmetry consistency (M2, M5)**: $L_{sym}​=∥f(x)−f(g(x))∥$

**TDA regularization (optional)**: $L_{TDA}​=∥ϕ(x)−ϕ(g(x))∥$

## 8. Evaluation Metrics
### 8.1 Standard ML metrics
* ROC-AUC
* Accuracy
* F1-score

### 8.2 Robustness metrics
(1) Group robustness

Evaluate: $R_G=E_{g∼G}[AUC(x,g(x))]$

(2) Learned group robustness: $R_{\hat G}$
	​
(3) Worst-case degradation: $Δ_{worst}​ = max_g(AUC_{clean}	​− AUC_g​)$

### 8.3 Topological metrics
(1) Persistence stability: $d_W​(D(x),D(gx))$

(Wasserstein distance between persistence diagrams)

(2) Betti curve drift: $∥β(x)−β(gx)∥$

(3) Topological variance ratio (TVR)

Fraction of variance explained by symmetry actions.

### 8.4 Equivariance diagnostics
 $E=∥ϕ(gx)−g′ϕ(x)∥$

## 9. Ablation Study Design

We explicitly isolate contributions:
### 9.1 Symmetry ablation
| Variant | G | Ĝ |
| ------- | - | - |
| A1      | ✗ | ✗ |
| A2      | ✔ | ✗ |
| A3      | ✗ | ✔ |
| A4      | ✔ | ✔ |
### 9.2 TDA ablation
| Variant | TDA              |
| ------- | ---------------- |
| B1      | ✗                |
| B2      | raw-space TDA    |
| B3      | latent-space TDA |
### 9.3 Fusion ablation
| Variant | Fusion                  |
| ------- | ----------------------- |
| F1      | concat                  |
| F2      | attention               |
| F3      | gated multimodal fusion |

## 10. Robustness Testing Protocol
### 10.1 Synthetic symmetry shifts
We apply controlled transformations:
* increasing rotation magnitude
* random jet permutations
* learned group perturbations

### 10.2 OOD shift test
* train on subset of feature space
* test on shifted distributions

### 10.3 Adversarial symmetry breaking
Construct worst-case transformation:
$g^{∗} = arg\ max_g ​L(f(x),f(g(x)))$

## 11. Statistical Testing
* report mean ± std over 5 seeds
* paired t-test between models
* bootstrap CI for AUC differences

Significance threshold:
* p < 0.05

## 12. Visualization Protocol
### 12.1 Embedding geometry
* UMAP of latent space
### 12.2 Orbit structure
* trajectories x → g(x)
### 12.3 TDA diagrams
* persistence diagrams per model
### 12.4 Symmetry-induced collapse plots
* manifold contraction under G vs Ĝ

## 13. Expected Results (paper narrative)

Typical outcomes you should expect:

M1 (hand-crafted)
* ↑ robustness
* limited ceiling in AUC
M2 (learned symmetry)
* ↑ AUC
* better OOD generalization
M3 (TDA only)
* moderate AUC
* strong anomaly sensitivity
M5 (learned + TDA)
* best trade-off:
    - highest robustness
    - stable topology
    - best calibration

## 14. Key Contribution Claim (NeurIPS framing)

You can phrase contributions like:
```
We propose a unified framework combining:

* symmetry discovery
* physics-informed augmentation
* topological representation learning

and demonstrate that learned symmetries reshape the topological structure of latent manifolds in a measurable and stable way.
```
Если хочешь следующий шаг, я могу дальше сделать одно из:

*  “NeurIPS paper draft” (Introduction + Related Work + Method)
* таблицу результатов (synthetic expected results + how to report)
* loss functions и training objective unified
* математическую теорию: “symmetry–topology coupling theorem (sketch)”

loss functions и training objective unified.

Ниже — единый (unified) training objective в стиле NeurIPS, который аккуратно объединяет:

* классификацию HIGGS
* hand-crafted symmetries
* learned symmetries (Atlas-style)
* TDA regularization
* equivariance / invariance constraints
* multimodal fusion

Я оформлю это как общую формулу + разложение по режимам + практическая реализация.

# 🧠 Unified Training Objective
## 1. Общая форма

Мы обучаем модель $f_θ​(classifier + encoder + fusion)$, минимизируя:
$$
L_{total} = L_{cls}	​+ λ_s * ​L_{sym} + λ_t * ​L_{TDA} ​+ λ_e * L_{equiv} ​ +λ_r * 	​L_{reg}
$$

## 2. Classification Loss

Стандартная бинарная кросс-энтропия:
$$
L_{cls}	​= −E(x,y)​[y*logf_θ(x)+(1−y)log(1−f_θ​(x))]
$$

## 3. Symmetry Consistency Loss
### 3.1 Hand-crafted symmetries

Для заданных групп G:
$$
L_{sym}^G = E_{x,g∼G}[∥f_θ​(x) − f_θ(gx)∥_2^2]
$$
👉 enforcing invariance

### 3.2 Learned symmetries (Atlas-style)

Для learned transformation $g_ϕ$:
$$
L_{sym}^{\hat G} = E_x	​[∥f_θ(x)−f_θ(g_ϕ(x))∥^2]
$$

### 3.3 Combined symmetry loss
$$
L_{sym}	​= αL_{sym}^G + (1−α)L_{sym}^{\hat G}
$$	​

## 4. Equivariance Loss (latent-level constraint)

Если есть encoder $z=h_θ(x)$:

$$
L_{equiv}​ = E_{x,g}[∥h_θ(gx)−ρ(g)h_θ(x)∥^2]
$$,где:
* ρ(g) — representation of group action in latent space
* может быть learned linear map

## 5. TDA Loss (topological regularization)
### 5.1 Idea

We enforce stability of topology under transformations.

### 5.2 Persistence diagram consistency

Let:
* PD(x) = persistence diagram
* Φ(PD) = vectorization (PI / Betti curves)
$$
L_{TDA}	​= E_{x,g}[d(Φ(PD(x)),Φ(PD(gx)))]
$$, 

where d is:
* Wasserstein distance (ideal)
* or L2 in persistence image space (practical)

### 5.3 Optional topology-aware smoothness
$$
L_{TDA} ​= ∥β(x)−β(gx)∥_2^2 + γ H(PD(x))
$$, 
(where entropy regularization stabilizes noisy topology)

## 6. Regularization Loss
Standard terms:
$$
L_{reg} = ∥θ∥_2^2 + dropout\ consistency\ penalty
$$

## 7. Final Unified Objective
### Full model:
$$
L_total​ = L_cls + λ_1 * ​L_{sym}^G + λ_2 * L_{sym}^{\hat G} + λ_3 * L_{equiv} + λ_4 * ​L_{TDA} + λ_5 * ∥θ∥^2
$$
​

## 8. Training regimes (important for NeurIPS clarity)

We define 3 operating modes:

1. Mode A — Physics-informed only

$$
L = L_{cls} + λ_1*L_{sym}^G
$$

 ✔ interpretable

 ✖ limited expressivity

2. Mode B — Learned symmetry only
$$
L = L_{cls}	​+ λ_2 * L_sym^{\hat G}​ + λ_3 * L_{equiv}
$$​

 ✔ adaptive invariance

 ✔ better OOD generalization

3.  Mode C — Full model (symmetry + TDA)

$$
L=L_{total}
$$​

 ✔ best robustness

 ✔ topology-stable representations

 ✔ strongest generalization

## 9. Implementation-friendly form (PyTorch-like pseudocode)

In [ ]:
loss_cls = BCE(pred, y)

loss_sym_g = ((f(x) - f(g(x)))**2).mean()
loss_sym_hatg = ((f(x) - f(g_theta(x)))**2).mean()

loss_equiv = ((enc(g(x)) - rho(g) @ enc(x))**2).mean()

loss_tda = wasserstein(persistence(x), persistence(g(x)))

loss_total = (
    loss_cls
    + l1 * loss_sym_g
    + l2 * loss_sym_hatg
    + l3 * loss_equiv
    + l4 * loss_tda
    + l5 * l2_regularization(model_params)
)

## 10. Key conceptual takeaway (NeurIPS framing)

Ты фактически оптимизируешь `a function that is simultaneously`:
* discriminative (classification)
* invariant/equivariant (symmetries)
* topology-stable (TDA consistency)

11. Если усиливать paper-идею, можно формально заявить “Symmetry–Topology Consistency Principle”
$$
g∈G⇒PD(x)≈PD(gx)
$$
и модель обучается приближать это равенство.

Если хочешь следующий шаг, я могу дальше сделать:

* “Method section” (полностью NeurIPS-ready текст)
* synthetic results table (как это будет выглядеть в paper)
* theoretical proposition: связь symmetry groups ↔ persistent homology invariants
* ablation narrative (как это пишут в NeurIPS rebuttal)

реальный Python PyTorch код всей системы

Ниже — реальный (но research-grade) PyTorch skeleton, который соответствует твоей системе:

* HIGGS classifier (MLP encoder + head)
* hand-crafted symmetry augmentation
* learned symmetry module (simple Atlas-style generator)
* TDA features (через ripser / persistence images stub)
* unified loss (symmetry + equivariance + TDA)
* training loop

Это не production, а NeurIPS-style experimental codebase skeleton, который можно расширять.

## 0. Dependencies

In [ ]:
pip install torch numpy scikit-learn ripser persim

## 1. Model Components
1.1 Encoder + Classifier

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, d_in=28, d_latent=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 128),
            nn.ReLU(),
            nn.Linear(128, d_latent),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


class Classifier(nn.Module):
    def __init__(self, d_latent=64):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(d_latent, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, z):
        return torch.sigmoid(self.head(z))

## 2. Hand-crafted symmetry augmentation

(упрощённая версия для tabular HIGGS)

In [ ]:
import torch
import random

def hand_crafted_symmetry(x):
    x = x.clone()

    # example: swap jet-like features
    if random.random() < 0.5:
        x[:, [4, 5]] = x[:, [5, 4]]

    # random sign flip (toy rotational symmetry)
    if random.random() < 0.5:
        x[:, 10:15] *= -1

    return x

## 3. Learned symmetry module (Atlas-style toy version)

In [ ]:
class SymmetryNet(nn.Module):
    """
    Learns transformation g_theta(x)
    """
    def __init__(self, d=28):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 128),
            nn.ReLU(),
            nn.Linear(128, d)
        )

    def forward(self, x):
        return x + 0.1 * self.net(x)  # residual transformation

## 4. TDA feature extractor (simplified)

In [ ]:
from ripser import ripser
import numpy as np

def compute_tda_features(x_numpy):
    """
    x_numpy: (batch, features)
    returns: persistence vector (toy version)
    """
    diagrams = ripser(x_numpy)['dgms']

    # very simplified: use number of features in H1
    h1 = diagrams[1]

    if len(h1) == 0:
        return np.array([0.0, 0.0])

    lifetimes = h1[:, 1] - h1[:, 0]
    return np.array([
        lifetimes.mean(),
        lifetimes.max()
    ])

## 5. Full Model Wrapper

In [ ]:
class FullModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.classifier = Classifier()
        self.sym_net = SymmetryNet()

    def forward(self, x):
        z = self.encoder(x)
        y = self.classifier(z)
        return y, z

    def transform(self, x):
        return self.sym_net(x)

## 6. Loss Functions

In [ ]:
def loss_cls(pred, y):
    return F.binary_cross_entropy(pred.squeeze(), y.float())


def loss_sym(f, x, x_aug):
    return ((f(x) - f(x_aug))**2).mean()


def loss_equiv(enc, x, x_aug, sym_net):
    return ((enc(x_aug) - enc(sym_net(x)))**2).mean()

## 7. Training Loop

In [ ]:
def train(model, loader, optimizer, lambda_sym=0.1, lambda_equiv=0.1):
    model.train()

    for x, y in loader:

        # -------------------------
        # original forward
        # -------------------------
        pred, z = model(x)

        # -------------------------
        # hand-crafted augmentation
        # -------------------------
        x_aug = hand_crafted_symmetry(x)

        pred_aug, z_aug = model(x_aug)

        # -------------------------
        # learned symmetry
        # -------------------------
        x_sym = model.transform(x)

        # -------------------------
        # losses
        # -------------------------
        L_cls = loss_cls(pred, y)

        L_sym = loss_sym(model.classifier, z, z_aug)

        L_equiv = loss_equiv(
            model.encoder,
            x,
            x_sym,
            model.sym_net
        )

        # -------------------------
        # total loss
        # -------------------------
        loss = (
            L_cls
            + lambda_sym * L_sym
            + lambda_equiv * L_equiv
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

## 8. Evaluation with TDA

In [ ]:
def evaluate_tda(model, X):
    model.eval()

    with torch.no_grad():
        z = model.encoder(X).cpu().numpy()

    tda_feat = compute_tda_features(z)

    return tda_feat

## 9. Full Experiment Runner

In [ ]:
def run_experiment(model, loader):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(10):
        train(model, loader, optimizer)

    return model

Что здесь уже реализовано (важно)
* ✔ encoder + classifier
* ✔ hand-crafted symmetry augmentation
* ✔ learned symmetry generator
* ✔ equivariance loss
* ✔ symmetry consistency loss
* ✔ hook for TDA features
* ✔ unified training loop

Чего пока нет (но это нормально для research skeleton)

Чтобы довести до NeurIPS-level system:

1. Proper TDA integration
* persistence images (Persim)
* differentiable TDA layer (advanced)
2. Real Atlas symmetry learning
* group manifold parameterization
* stochastic Lie algebra model
3. Better equivariance
* explicit group representation ρ(g)
4. Multi-view contrastive learning

Если хочешь следующий шаг, я могу превратить это в:

1. Full research codebase structure
```
project/
  models/
  losses/
  symmetry/
  tda/
  train.py
  eval.py
```

2. Or “NeurIPS submission-ready Method + Code appendix”

3. Or upgrade to:

👉 equivariant neural network + differentiable TDA (real SOTA version)

# превратить это в: 1. Full research codebase structure

Да. Если ориентироваться на уровень **NeurIPS/ICML/ICLR reproducibility**, то я бы уже не делал всё в одном `train.py`. Лучше сразу организовать код как полноценный исследовательский фреймворк, где можно легко запускать абляции (`baseline`, `hand-crafted`, `Atlas`, `TDA`, `Atlas+TDA`).

Я бы предложил примерно такую структуру.

```text
higgs-symmetry-tda/
│
├── README.md
├── pyproject.toml
├── requirements.txt
├── setup.cfg
│
├── configs/
│   ├── default.yaml
│   ├── baseline.yaml
│   ├── handcrafted.yaml
│   ├── atlas.yaml
│   ├── tda.yaml
│   └── full.yaml
│
├── data/
│   ├── download_higgs.py
│   ├── preprocess.py
│   ├── dataset.py
│   └── splits.py
│
├── models/
│   ├── encoder.py
│   ├── classifier.py
│   ├── fusion.py
│   ├── symmetry_net.py
│   ├── full_model.py
│   └── __init__.py
│
├── symmetry/
│   ├── handcrafted.py
│   ├── atlas.py
│   ├── lie_group.py
│   ├── augment.py
│   └── metrics.py
│
├── tda/
│   ├── persistence.py
│   ├── vectorization.py
│   ├── wasserstein.py
│   ├── diagrams.py
│   └── features.py
│
├── losses/
│   ├── classification.py
│   ├── symmetry.py
│   ├── equivariance.py
│   ├── topology.py
│   ├── contrastive.py
│   └── total.py
│
├── trainers/
│   ├── trainer.py
│   ├── callbacks.py
│   ├── scheduler.py
│   └── checkpoint.py
│
├── evaluation/
│   ├── metrics.py
│   ├── robustness.py
│   ├── calibration.py
│   ├── topology.py
│   └── statistics.py
│
├── visualization/
│   ├── latent.py
│   ├── persistence.py
│   ├── betti.py
│   ├── symmetry.py
│   └── paper_figures.py
│
├── experiments/
│   ├── exp01_baseline.py
│   ├── exp02_handcrafted.py
│   ├── exp03_atlas.py
│   ├── exp04_tda.py
│   ├── exp05_full.py
│   └── ablations.py
│
├── scripts/
│   ├── train.py
│   ├── evaluate.py
│   ├── inference.py
│   └── reproduce_paper.py
│
├── notebooks/
│   ├── 01_EDA.ipynb
│   ├── 02_TDA.ipynb
│   ├── 03_Symmetry.ipynb
│   └── 04_PaperPlots.ipynb
│
├── tests/
│   ├── test_losses.py
│   ├── test_symmetry.py
│   ├── test_tda.py
│   └── test_model.py
│
└── outputs/
    ├── checkpoints/
    ├── logs/
    ├── figures/
    └── tables/
```

---

# Что делает каждая директория

## `configs/`

Все параметры экспериментов.

Например

```yaml
model:
    latent_dim: 128

tda:
    enabled: true
    homology_dim: 1

symmetry:
    handcrafted: true
    atlas: true

loss:
    lambda_sym: 0.2
    lambda_tda: 0.05
```

Таким образом переключение между экспериментами происходит без изменения кода.

---

# `data/`

Отвечает только за данные.

```
download_higgs.py
```

скачивает датасет.

```
preprocess.py
```

* normalization
* train/test split
* feature engineering

```
dataset.py
```

PyTorch Dataset.

---

# `models/`

Только нейронные сети.

```
Encoder
Classifier
Fusion
SymmetryNet
```

Никаких функций потерь здесь нет.

---

# `symmetry/`

Это отдельный модуль.

```
handcrafted.py
```

Например

```python
rotate_phi(...)
swap_jets(...)
reflection(...)
```

---

```
atlas.py
```

реализует

```python
AtlasDiscovery()

AtlasSampler()

OrbitGenerator()
```

---

```
lie_group.py
```

Если потом захочешь использовать

* SO(2)
* SO(3)
* SE(3)

или их приближения.

---

# `tda/`

Отдельная библиотека.

```
persistence.py
```

строит

```
Ripser
Gudhi
```

---

```
vectorization.py
```

делает

* persistence image
* persistence landscape
* persistence entropy

---

```
features.py
```

возвращает обычный

```python
Tensor
```

который можно подавать в MLP.

---

# `losses/`

Очень удобно.

Каждая функция отдельно.

```
classification.py
```

```python
BCELoss
```

---

```
symmetry.py
```

```python
InvariantLoss
```

---

```
equivariance.py
```

```python
EquivariantLoss
```

---

```
topology.py
```

```python
PersistenceLoss
```

---

```
total.py
```

собирает

[
L=L_{cls}+\lambda_1L_{sym}
+\lambda_2L_{eq}
+\lambda_3L_{tda}
]

---

# `trainers/`

Здесь живёт цикл обучения.

```
Trainer.fit()
```

```
Trainer.validate()
```

```
Trainer.test()
```

---

# `evaluation/`

Отдельно считается

```
ROC

AUC

F1

Calibration

OOD

Robustness
```

и

```
Wasserstein distance

Betti curves

Topology stability
```

---

# `visualization/`

Именно отсюда строятся рисунки статьи.

Например

```
paper_figures.py
```

генерирует

```
Figure 2

Figure 3

Figure 4
```

автоматически.

---

# `experiments/`

Это одна из самых важных директорий.

Каждый файл соответствует одной таблице статьи.

Например

```
exp01_baseline.py
```

получает

Table 1

---

```
exp02_handcrafted.py
```

получает

Table 2

---

```
ablations.py
```

строит

Ablation Study.

---

# `scripts/`

Командная строка.

Например

```
python train.py \
    --config configs/full.yaml
```

или

```
python evaluate.py \
    --checkpoint model.pt
```

---

# `outputs/`

Стандартная структура

```
outputs/

    checkpoints/

    tensorboard/

    logs/

    tables/

    figures/
```

---

# Как это будет выглядеть в статье

Такой проект естественным образом соответствует разделам статьи:

| Раздел статьи         | Код              |
| --------------------- | ---------------- |
| 3.1 Dataset           | `data/`          |
| 3.2 Architecture      | `models/`        |
| 3.3 Symmetry Learning | `symmetry/`      |
| 3.4 TDA               | `tda/`           |
| 3.5 Loss Function     | `losses/`        |
| 4. Experiments        | `experiments/`   |
| 5. Evaluation         | `evaluation/`    |
| Figures               | `visualization/` |

---

## Что ещё можно улучшить

Если цель — действительно сильная исследовательская работа, я бы добавил:

* **Hydra** или аналогичную систему управления конфигурациями для воспроизводимых запусков.
* **PyTorch Lightning** или **Lightning Fabric** для стандартизации цикла обучения и логирования.
* Логирование через **Weights & Biases** или **MLflow**.
* Автоматический запуск серий экспериментов (sweeps) по различным значениям коэффициентов потерь, параметрам аугментаций и TDA.
* Отдельный модуль для **статистического анализа** (bootstrap, доверительные интервалы, тесты значимости), чтобы результаты в статье были не только точечными, но и статистически обоснованными.

Такая организация позволит без существенных изменений расширять проект, например, заменять HIGGS на другие HEP-датасеты или сравнивать несколько методов поиска симметрий в одном экспериментальном фреймворке.


# Автоматический запуск серий экспериментов (sweeps) по различным значениям коэффициентов потерь, параметрам аугментаций и TDA.

Для статьи уровня **NeurIPS** я бы вообще не запускал эксперименты вручную. Я бы строил полноценный **Experiment Manager**, который автоматически перебирает конфигурации, сохраняет результаты в едином формате и затем автоматически генерирует все таблицы и рисунки статьи.

Архитектура выглядела бы примерно так:

```text
configs/
    baseline.yaml
    handcrafted.yaml
    atlas.yaml
    tda.yaml
    full.yaml

sweeps/
    lambda_search.yaml
    augmentations.yaml
    tda.yaml
    architecture.yaml
    paper.yaml

experiments/
    train.py
    evaluate.py

outputs/
    run_001/
    run_002/
    ...
```

---

# 1. Конфигурации экспериментов

Например, базовый `configs/full.yaml`

```yaml
experiment:
    name: full

seed: 42

model:
    latent_dim: 128
    hidden_dim: 256

training:
    epochs: 100
    batch_size: 512
    lr: 1e-3

loss:
    lambda_cls: 1.0
    lambda_sym: 0.2
    lambda_equiv: 0.1
    lambda_tda: 0.05

symmetry:
    handcrafted: true
    atlas: true

tda:
    enabled: true
    homology_dim: 1
```

---

# 2. Sweep по коэффициентам потерь

Например

```yaml
defaults:
  - override hydra/sweeper: optuna

hydra:

  sweeper:

    direction: maximize

    study_name: loss_search

    n_trials: 100

    params:

      loss.lambda_sym: interval(0.0,0.5)

      loss.lambda_equiv: interval(0.0,0.5)

      loss.lambda_tda: interval(0.0,0.3)
```

Тогда автоматически будут обучены десятки моделей.

---

# 3. Sweep по hand-crafted augmentation

```yaml
augmentation:

    rotation_probability:

        values:
            - 0.0
            - 0.25
            - 0.5
            - 0.75
            - 1.0

    permutation_probability:

        values:
            - 0.0
            - 0.5
            - 1.0

    sign_flip:

        values:
            - true
            - false
```

---

# 4. Sweep по Atlas

```yaml
atlas:

    latent_dim:

        values:
            - 16
            - 32
            - 64
            - 128

    hidden_layers:

        values:
            - 2
            - 3
            - 4

    residual_strength:

        values:
            - 0.05
            - 0.1
            - 0.2
```

---

# 5. Sweep по TDA

```yaml
tda:

    homology:

        values:
            - 0
            - 1
            - [0,1]

    vectorization:

        values:

            - persistence_image

            - persistence_landscape

            - persistence_entropy

            - betti_curve

    sigma:

        values:

            - 0.05

            - 0.1

            - 0.2
```

---

# 6. Sweep по архитектуре

```yaml
model:

    latent_dim:

        values:

            - 64

            - 128

            - 256

    hidden_dim:

        values:

            - 128

            - 256

            - 512

    dropout:

        values:

            - 0.0

            - 0.2

            - 0.5
```

---

# 7. Полный paper sweep

Именно его обычно запускают на кластере.

```yaml
experiment:

    seed:

        values:

            - 0
            - 1
            - 2
            - 3
            - 4

loss:

    lambda_sym:

        values:

            - 0.0
            - 0.1
            - 0.2

    lambda_tda:

        values:

            - 0.0
            - 0.05
            - 0.1

symmetry:

    atlas:

        values:

            - true
            - false

tda:

    enabled:

        values:

            - true
            - false
```

Даже такой сравнительно небольшой набор даёт:

* 5 seeds × 3 значения λ_sym × 3 значения λ_tda × 2 режима Atlas × 2 режима TDA = **180 запусков**.

---

# 8. Автоматическое сохранение результатов

После каждого запуска сохраняется JSON.

```json
{
  "seed": 2,
  "auc": 0.851,
  "accuracy": 0.804,
  "f1": 0.791,
  "lambda_sym": 0.2,
  "lambda_tda": 0.05,
  "atlas": true,
  "tda": true
}
```

или CSV

```text
run_id,seed,auc,f1,lambda_sym,lambda_tda,atlas,tda
001,0,0.847,0.786,0.1,0.05,True,True
002,1,0.849,0.789,0.1,0.05,True,True
...
```

---

# 9. Автоматическая агрегация

После завершения всех запусков выполняется один скрипт:

```bash
python aggregate_results.py
```

Он строит итоговые таблицы.

Например:

| Model        |             AUC |              F1 | Robustness |
| ------------ | --------------: | --------------: | ---------: |
| Baseline     |     0.842±0.002 |     0.781±0.003 |       0.74 |
| Hand-crafted |     0.848±0.002 |     0.786±0.002 |       0.81 |
| Atlas        |     0.854±0.002 |     0.792±0.002 |       0.86 |
| Atlas+TDA    | **0.861±0.001** | **0.801±0.002** |   **0.91** |

---

# 10. Автоматическая генерация рисунков

Из тех же результатов можно автоматически строить:

* **heatmap** зависимости AUC от (\lambda_{\text{sym}}) и (\lambda_{\text{tda}});
* **boxplot** по различным random seeds;
* **Pareto-фронт** Accuracy ↔ Robustness;
* **кривые обучения** (train/validation loss);
* **scatter** "Topological Stability vs AUC";
* **UMAP** латентных представлений до и после применения learned symmetries;
* **диаграммы устойчивости** (persistence diagrams) и их изменение под действием hand-crafted и learned преобразований.

---

## Следующий шаг для полноценного исследовательского проекта

Для такой работы я бы рекомендовал использовать следующий стек:

* **Hydra** — управление конфигурациями и мультизапусками.
* **Optuna** — автоматическая оптимизация гиперпараметров (коэффициенты потерь, параметры TDA, архитектуры).
* **PyTorch Lightning** — стандартизированный цикл обучения и удобное сохранение чекпоинтов.
* **Weights & Biases** или **MLflow** — централизованное логирование всех 100–1000 запусков, сравнение моделей и автоматическое построение дашбордов.
* **Pandas + Seaborn/Matplotlib** — автоматическая агрегация результатов и генерация таблиц/рисунков, которые затем можно практически без изменений вставлять в статью. Это позволит воспроизводить все эксперименты одной командой и существенно упростит подготовку публикации.


Это именно тот стек, который я бы рекомендовал для исследования уровня **NeurIPS/ICML**. Но я бы пошёл ещё дальше и построил проект по принципу **Research Operating System (Research OS)**, где любая фигура, таблица или результат статьи автоматически воспроизводятся из конфигурации эксперимента.

Ниже — архитектура такого проекта.

---

# Общая архитектура

```text
                +---------------------------+
                |      Hydra Configs        |
                +------------+--------------+
                             |
                             v
                +---------------------------+
                |     Experiment Runner     |
                |  (Lightning + Hydra)      |
                +------------+--------------+
                             |
          +------------------+------------------+
          |                  |                  |
          v                  v                  v
   Symmetry Module      TDA Module        Model Module
(hand-crafted/Atlas)   (Ripser/GUDHI)     (MLP/Fusion)
          |                  |                  |
          +------------------+------------------+
                             |
                             v
                  Unified Loss Function
                             |
                             v
                  Lightning Trainer
                             |
         +-------------------+------------------+
         |                                      |
         v                                      v
Weights & Biases / MLflow              Checkpoints
         |                                      |
         +-------------------+------------------+
                             |
                             v
                  Result Aggregator
                             |
         +-------------------+------------------+
         |                  |                  |
         v                  v                  v
     Tables           Paper Figures      Statistical Tests
```

---

# Полная структура проекта

```text
higgs_symmetry_tda/

├── configs/
│
│   dataset/
│   model/
│   training/
│   symmetry/
│   tda/
│   optimizer/
│   logger/
│   experiment/
│
├── src/
│
│   data/
│
│   models/
│
│   symmetry/
│
│   tda/
│
│   losses/
│
│   callbacks/
│
│   trainers/
│
│   evaluation/
│
│   visualization/
│
│   utils/
│
├── sweeps/
│
├── scripts/
│
├── notebooks/
│
├── outputs/
│
└── paper/
```

---

# Hydra

Hydra становится центральной системой управления.

Структура:

```text
configs/

    config.yaml

    model/

        mlp.yaml

        transformer.yaml

    symmetry/

        handcrafted.yaml

        atlas.yaml

    tda/

        persistence_image.yaml

        landscape.yaml

        entropy.yaml

    optimizer/

        adam.yaml

        adamw.yaml
```

Главный конфиг

```yaml
defaults:

- model: mlp
- symmetry: atlas
- tda: persistence_image
- optimizer: adam
```

Теперь переключение модели выглядит так:

```bash
python train.py model=transformer
```

или

```bash
python train.py symmetry=handcrafted
```

без изменения кода.

---

# PyTorch Lightning

Вся логика обучения помещается в один класс.

```python
class HiggsLightningModule(L.LightningModule):

    def training_step(...)

    def validation_step(...)

    def configure_optimizers(...)
```

Lightning автоматически обеспечивает:

* mixed precision
* multi-GPU
* gradient clipping
* checkpointing
* early stopping
* learning rate scheduler

Практически исчезает необходимость писать собственный цикл обучения.

---

# Optuna

Hydra напрямую интегрируется с Optuna.

Например,

```yaml
hydra:

  sweeper:

    sampler:

      _target_: optuna.samplers.TPESampler
```

Далее

```yaml
params:

    loss.lambda_sym: interval(0.0,0.4)

    loss.lambda_tda: interval(0.0,0.3)

    model.latent_dim: choice(64,128,256)

    training.lr: tag(log, interval(1e-5,1e-2))
```

После запуска

```bash
python train.py --multirun
```

Optuna сама построит последовательность экспериментов.

---

# Weights & Biases

Каждый запуск автоматически логирует

```text
loss

validation loss

AUC

F1

ROC

confusion matrix

learning rate

lambda_sym

lambda_tda

Atlas latent dimension

TDA dimension

GPU utilization

wall-clock time
```

Кроме того, можно сохранять артефакты:

```text
Persistence diagrams

UMAP

Betti curves

Model checkpoints
```

Все они становятся доступны через веб-интерфейс.

---

# MLflow (альтернатива)

Если проект выполняется в закрытой инфраструктуре, MLflow предоставляет:

```
Experiment

Run

Parameters

Metrics

Artifacts

Model Registry
```

Это особенно удобно при развёртывании собственного сервера.

---

# Callback система Lightning

Например,

```text
callbacks/

    checkpoint.py

    early_stopping.py

    tda_visualization.py

    symmetry_visualization.py

    wandb_logger.py
```

Во время обучения можно автоматически:

каждые N эпох

↓

строить

Persistence Diagram

↓

сохранять

↓

отправлять в W&B.

---

# Sweep Pipeline

```
Hydra

↓

создает конфигурацию

↓

Optuna

↓

выбирает параметры

↓

Lightning

↓

обучает модель

↓

W&B

↓

логирует результаты

↓

Result Aggregator

↓

обновляет таблицы статьи
```

---

# Автоматическая генерация paper figures

После завершения всех запусков

```bash
python scripts/build_paper.py
```

автоматически появляются

```text
paper/

    fig1_pipeline.pdf

    fig2_auc_vs_lambda.pdf

    fig3_topology.pdf

    fig4_umap.pdf

    fig5_ablation.pdf

    table1.tex

    table2.tex

    table3.tex
```

Таблицы сразу сохраняются в LaTeX.

Например

```latex
\begin{tabular}
...
\end{tabular}
```

их можно просто вставить в статью.

---

# Автоматическая статистика

Отдельный модуль

```text
statistics/

    bootstrap.py

    significance.py

    effect_size.py
```

считает

* bootstrap confidence intervals

* paired t-test

* Wilcoxon signed-rank

* Cohen's d

Получаются готовые значения

```
AUC

0.861 ± 0.002

p=0.004
```

которые сразу используются в статье.

---

# Полностью воспроизводимая статья

Самая сильная часть такого подхода — возможность воспроизвести результаты одной командой.

```bash
python scripts/reproduce_paper.py
```

Этот сценарий последовательно:

1. скачивает и подготавливает данные;
2. запускает все конфигурации через Hydra и Optuna;
3. обучает модели с помощью Lightning;
4. логирует результаты в W&B или MLflow;
5. агрегирует метрики;
6. выполняет статистические тесты;
7. строит все рисунки;
8. генерирует таблицы в формате LaTeX;
9. сохраняет итоговые артефакты в каталог `paper/`.

Такой процесс соответствует современным требованиям к воспроизводимости исследований и значительно упрощает подготовку публикации. Для проектов такого масштаба (сотни запусков и множественные абляции) особенно важно также фиксировать версии библиотек, использовать детерминированные random seeds (насколько это возможно) и документировать ограничения воспроизводимости, связанные с аппаратным обеспечением или недетерминированными CUDA-операциями.


# довести до NeurIPS-level system:

1. Proper TDA integration
persistence images (Persim)
differentiable TDA layer (advanced)
2. Real Atlas symmetry learning
group manifold parameterization
stochastic Lie algebra model
3. Better equivariance
explicit group representation ρ(g)
4. Multi-view contrastive learning

Это как раз тот момент, где заканчивается "хороший ML-проект" и начинается **исследование уровня NeurIPS**. Но здесь есть важный нюанс: некоторые пункты (например, "Real Atlas symmetry learning") не являются стандартными компонентами с общепринятой реализацией. Их придется проектировать самостоятельно или адаптировать идеи из нескольких работ. Поэтому ниже я разделю, что является **устоявшейся практикой**, а что — **исследовательским предложением**.

Я бы предложил следующую архитектуру.

---

# Общая математическая модель

Пусть

[
x\in\mathcal X
]

— событие HIGGS,

[
z=f_\theta(x)
]

— латентное представление,

[
g\in G
]

— элемент некоторой группы преобразований.

Мы хотим одновременно обучить

[
f_\theta,\qquad
G,\qquad
\rho,\qquad
TDA
]

в рамках одной оптимизации.

Получается

```text
Raw Event
      │
      ▼
Encoder
      │
      ├──────────────┐
      │              │
      ▼              ▼
Latent z       Symmetry Generator
      │              │
      │              ▼
      │         g(z)
      │              │
      ├──────────────┤
      ▼
Multi-view Contrastive Module
      │
      ▼
Differentiable TDA Layer
      │
      ▼
Fusion
      │
      ▼
Classifier
```

---

# 1. Proper TDA Integration

Это наиболее зрелая часть всей системы.

## Архитектура

```text
latent embedding

↓

Ripser / GUDHI

↓

Persistence Diagram

↓

Persistence Image

↓

CNN / MLP

↓

Topological embedding
```

---

## Persistence Image

Вместо

```python
mean lifetime
```

используется полноценная

[
PI(x)\in\mathbb R^{m\times n}
]

которая строится как

[
PI(u,v)
=======

\sum_i
w_i
\exp
\left(
-\frac{
|(u,v)-p_i|^2
}{
2\sigma^2
}
\right)
]

где

(p_i)

—

точки persistence diagram.

---

### Затем

```text
Persistence Image

↓

Conv2D

↓

Embedding

↓

Fusion
```

---

## Differentiable TDA

Вот здесь начинается NeurIPS.

Вместо

```
Ripser

↓

detach()
```

используется

```
Differentiable Persistence Layer
```

которая позволяет распространять градиенты.

Возможные варианты реализации:

* дифференцируемые слои поверх persistent homology;
* сглаженные (soft) аппроксимации persistence;
* методы из библиотек, поддерживающих обратное распространение для топологических характеристик.

Это уже полноценная исследовательская составляющая.

---

# 2. Real Atlas Symmetry Learning

Это самая интересная часть.

Вместо

```python
MLP(x)
```

мы хотим получить

[
g_\theta
========

\exp(A_\theta)
]

где

[
A_\theta
]

—

элемент алгебры Ли.

---

## Lie algebra parameterization

Параметризуем

[
A
=

\sum_i
\alpha_i B_i
]

где

(B_i)

—

базис генераторов.

Например

```text
SO(2)

↓

one generator

SO(3)

↓

three generators

SE(3)

↓

six generators
```

---

## Затем

через

[
g=\exp(A)
]

получаем группу.

---

То есть сеть учит

не преобразование

а

генераторы.

---

## Стохастическая модель

Можно ввести

[
\alpha_i
\sim
\mathcal N(\mu_i,\sigma_i)
]

Получаем

```
latent

↓

Lie algebra

↓

sample coefficients

↓

matrix exponential

↓

group element

↓

augmentation
```

Это уже похоже на "Atlas-like" обучение скрытых симметрий, но это не стандартный алгоритм — его нужно будет чётко описать и экспериментально обосновать.

---

# 3. Explicit group representation

Вместо

```
enc(gx)

≈

enc(x)
```

используем

[
f(gx)
=====

\rho(g)
f(x)
]

где

[
\rho(g)
]

—

представление группы.

---

## Например

пусть

[
z
=

(z_1,z_2,z_3)
]

Тогда

[
\rho(g)
=======

\begin{pmatrix}
R &0\
0&I
\end{pmatrix}
]

где

R

—

rotation matrix.

---

Loss

[
L
=

|
f(gx)
-----

\rho(g)f(x)
|^2
]

становится настоящим equivariance loss.

---

# 4. Multi-view Contrastive Learning

Это сегодня практически обязательный компонент.

---

Вместо одной аугментации

делаем четыре.

```text
x

↓

hand-crafted

↓

view1
```

```text
x

↓

Atlas

↓

view2
```

```text
x

↓

TDA perturbation

↓

view3
```

```text
x

↓

dropout

↓

view4
```

---

Encoder получает

четыре представления

```
z1

z2

z3

z4
```

---

Используем InfoNCE

[
L_{InfoNCE}
===========

*

\log
\frac{
\exp(sim(z_i,z_j)/\tau)
}{
\sum_k
\exp(sim(z_i,z_k)/\tau)
}
]

---

Теперь

TDA

становится ещё одной

view.

---

То есть

```
raw

↓

encoder

↓

embedding
```

и

```
Persistence Image

↓

CNN

↓

embedding
```

должны совпадать.

---

Получается дополнительный loss

[
L_{topo}
========

|
z-
z_{tda}
|^2
]

или его контрастивная версия.

---

# Финальная функция потерь

В результате можно использовать следующую композицию:

[
\mathcal L=
\lambda_{cls}\mathcal L_{cls}
+\lambda_{sym}\mathcal L_{sym}
+\lambda_{eq}\mathcal L_{equiv}
+\lambda_{ctr}\mathcal L_{InfoNCE}
+\lambda_{tda}\mathcal L_{tda}
+\lambda_{reg}\mathcal L_{reg}.
]

Такое разложение позволяет проводить чистые абляции (например, отключать только контрастивную часть или только топологическую регуляризацию), что особенно важно для публикации.

---

## Что отличает такую систему от обычного ML-пайплайна

С научной точки зрения наиболее оригинальной частью является **совместное обучение трёх структур**:

* **группы преобразований** (через параметризацию генераторов и представление группы),
* **латентного пространства**, которое удовлетворяет условию эквивариантности,
* **топологических инвариантов**, вычисляемых на этих представлениях и включённых в оптимизацию.

Именно взаимодействие этих компонентов — а не каждый из них по отдельности — может стать центральным исследовательским вкладом статьи. При этом стоит учитывать, что наиболее рискованные части (например, стохастическая параметризация алгебры Ли и полностью дифференцируемая TDA) потребуют особенно тщательной экспериментальной проверки и сильных абляций, чтобы убедительно показать их вклад.


# превратить в Python PyTorch код всей системы

Да. На первом этапе имеет смысл построить **каркас исследовательского фреймворка**, не зависящий от конкретной модели. После этого все остальные компоненты (Atlas, TDA, Contrastive Learning и т.д.) будут подключаться как плагины.

Ниже приведена архитектура **этапа 1**.

---

# Архитектура

```text id="m8p5cl"
higgs_symmetry_tda/
│
├── pyproject.toml
├── README.md
├── requirements.txt
│
├── configs/
│
│   config.yaml
│
│   experiment/
│       baseline.yaml
│       full.yaml
│
│   dataset/
│       higgs.yaml
│
│   model/
│       mlp.yaml
│
│   optimizer/
│       adam.yaml
│
│   trainer/
│       default.yaml
│
│   logger/
│       wandb.yaml
│
│   callbacks/
│       default.yaml
│
├── src/
│
│   train.py
│
│   datamodule.py
│
│   lightning_module.py
│
│   models/
│
│   callbacks/
│
│   utils/
│
└── outputs/
```

---

# requirements.txt

```text id="op94mj"
torch

lightning

hydra-core

omegaconf

wandb

optuna

pandas

numpy

scikit-learn

matplotlib

seaborn
```

---

# pyproject.toml

```toml id="0lxj3v"
[project]

name = "higgs_symmetry_tda"

version = "0.1.0"
```

---

# Hydra

## config.yaml

```yaml id="5pxwjq"
defaults:

  - dataset: higgs

  - model: mlp

  - optimizer: adam

  - trainer: default

  - logger: wandb

  - callbacks: default

  - experiment: baseline
```

---

# configs/model/mlp.yaml

```yaml id="n2w66o"
input_dim: 28

hidden_dim: 256

latent_dim: 128

dropout: 0.2
```

---

# configs/optimizer/adam.yaml

```yaml id="lqzmv5"
_target_: torch.optim.Adam

lr: 0.001

weight_decay: 0.0001
```

---

# trainer/default.yaml

```yaml id="zr4j46"
max_epochs: 100

accelerator: auto

devices: 1

precision: 32

gradient_clip_val: 1.0
```

---

# callbacks/default.yaml

```yaml id="p00mkc"
checkpoint: true

early_stopping: true

lr_monitor: true
```

---

# logger/wandb.yaml

```yaml id="qdkcxa"
project: higgs_symmetry_tda

entity: null

save_dir: outputs
```

---

# datamodule.py

Пока только каркас.

```python id="6qitwl"
import lightning as L

class HiggsDataModule(L.LightningDataModule):

    def __init__(self, cfg):

        super().__init__()

        self.cfg = cfg

    def prepare_data(self):

        pass

    def setup(self, stage=None):

        pass

    def train_dataloader(self):

        pass

    def val_dataloader(self):

        pass

    def test_dataloader(self):

        pass
```

---

# models/mlp.py

```python id="zphjpd"
import torch.nn as nn


class MLP(nn.Module):

    def __init__(self, cfg):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(cfg.input_dim, cfg.hidden_dim),

            nn.ReLU(),

            nn.Dropout(cfg.dropout),

            nn.Linear(cfg.hidden_dim, cfg.latent_dim),

            nn.ReLU(),

            nn.Linear(cfg.latent_dim,1)

        )

    def forward(self,x):

        return self.net(x)
```

---

# lightning_module.py

```python id="g2cqqq"
import lightning as L

import torch

import torch.nn.functional as F

from src.models.mlp import MLP


class HiggsModule(L.LightningModule):

    def __init__(self,cfg):

        super().__init__()

        self.save_hyperparameters()

        self.cfg=cfg

        self.model=MLP(cfg.model)

    def forward(self,x):

        return self.model(x)

    def training_step(self,batch,batch_idx):

        x,y=batch

        logits=self(x).squeeze()

        loss=F.binary_cross_entropy_with_logits(

            logits,

            y.float()

        )

        self.log("train_loss",loss)

        return loss

    def validation_step(self,batch,batch_idx):

        x,y=batch

        logits=self(x).squeeze()

        loss=F.binary_cross_entropy_with_logits(

            logits,

            y.float()

        )

        self.log("val_loss",loss)

    def configure_optimizers(self):

        return torch.optim.Adam(

            self.parameters(),

            lr=self.cfg.optimizer.lr,

            weight_decay=self.cfg.optimizer.weight_decay

        )
```

---

# train.py

```python id="x0ijyr"
import hydra

import lightning as L

from hydra.utils import instantiate

from omegaconf import DictConfig

from lightning.pytorch.loggers import WandbLogger

from src.datamodule import HiggsDataModule

from src.lightning_module import HiggsModule


@hydra.main(

    version_base=None,

    config_path="../configs",

    config_name="config"

)

def main(cfg: DictConfig):

    logger=WandbLogger(

        project=cfg.logger.project

    )

    dm=HiggsDataModule(cfg)

    model=HiggsModule(cfg)

    trainer=L.Trainer(

        max_epochs=cfg.trainer.max_epochs,

        accelerator=cfg.trainer.accelerator,

        devices=cfg.trainer.devices,

        logger=logger

    )

    trainer.fit(

        model,

        datamodule=dm

    )


if __name__=="__main__":

    main()
```

---

# Запуск

Обычный

```bash id="rjlwm0"
python src/train.py
```

---

С другой моделью

```bash id="jlwmxj"
python src/train.py model=transformer
```

---

С другим learning rate

```bash id="c6hvm7"
python src/train.py optimizer.lr=1e-4
```

---

С несколькими конфигурациями

```bash id="49l5pm"
python src/train.py --multirun optimizer.lr=1e-3,1e-4,1e-5
```

---

# Что уже готово после этапа 1

На этом этапе получается **рабочий исследовательский каркас**, в котором:

* конфигурации централизованы через Hydra;
* обучение стандартизировано с помощью PyTorch Lightning;
* логирование поддерживает W&B (при необходимости легко заменить на MLflow);
* структура проекта готова для подключения новых моделей, модулей симметрий, TDA и дополнительных функций потерь;
* можно запускать серии экспериментов с различными параметрами без изменения исходного кода.

Это создаёт основу, на которую последовательно можно добавлять более сложные исследовательские компоненты. Следующим логичным этапом будет полноценный `HiggsDataModule` с загрузкой, предобработкой и воспроизводимыми разбиениями данных, чтобы весь цикл обучения уже работал на реальном датасете HIGGS.


# 2 этап - DataModule для HIGGS + preprocessing

Отлично. На **этапе 2** мы переходим от каркаса к полноценному **Data Layer**, который должен соответствовать требованиям воспроизводимых исследований.

Я бы не делал DataModule "только под HIGGS". Лучше сразу сделать его расширяемым, чтобы впоследствии можно было заменить HIGGS на SUSY, MiniBooNE, HEPMASS и другие HEP-датасеты практически без изменения кода.

---

# Архитектура Data Layer

```text id="9gjf4z"
src/data/

    __init__.py

    base.py
    higgs.py
    preprocessing.py
    transforms.py
    split.py
    download.py
    statistics.py
    cache.py

configs/

    dataset/

        higgs.yaml

data/

    raw/
    processed/
    cache/
```

---

# Структура данных

```text id="3c1qpn"
data/

    raw/

        HIGGS.csv.gz

    processed/

        train.parquet

        val.parquet

        test.parquet

    cache/

        scaler.pkl

        statistics.json
```

---

# Конфигурация Hydra

`configs/dataset/higgs.yaml`

```yaml id="0jld5q"
name: higgs

root: data

download: true

batch_size: 512

num_workers: 8

pin_memory: true

persistent_workers: true

train_ratio: 0.70

val_ratio: 0.15

test_ratio: 0.15

seed: 42

normalize: standard

cache: true
```

---

# BaseDataModule

Все датасеты наследуются от одного класса.

```python id="72kgpq"
class BaseDataModule(L.LightningDataModule):

    def prepare_data(self):
        raise NotImplementedError

    def setup(self, stage=None):
        raise NotImplementedError

    def train_dataloader(self):
        raise NotImplementedError

    def val_dataloader(self):
        raise NotImplementedError

    def test_dataloader(self):
        raise NotImplementedError
```

---

# Download Module

```text id="lvwl4t"
download.py
```

```python id="szbxdr"
def download_higgs(root):

    if dataset_exists(root):

        return

    ...

    download()

    verify_checksum()
```

Лучше сразу предусмотреть:

* проверку SHA256
* возможность повторного использования уже скачанных данных
* автоматическую распаковку.

---

# Preprocessing Pipeline

Я бы разделил его на независимые шаги.

```text id="1br6ko"
Raw CSV

↓

Load

↓

Feature validation

↓

Missing values

↓

Outlier clipping

↓

Normalization

↓

Split

↓

Save parquet
```

---

# preprocessing.py

Каждый шаг оформляется отдельной функцией.

```python id="1dkklv"
validate_features()

remove_nan()

clip_outliers()

normalize()

save()
```

---

# Валидация признаков

Проверяем:

```python id="mllrzc"
assert X.shape[1] == 28

assert labels.unique() == {0,1}
```

---

# Обработка NaN

Для HIGGS обычно NaN отсутствуют, но общий код лучше оставить.

```python id="dixyha"
SimpleImputer()

или

dropna()
```

---

# Outlier Clipping

Для физических данных полезно использовать

```python id="b7o4fm"
Quantile clipping

0.5%

99.5%
```

вместо обычного

```id="gn3l2t"
StandardScaler
```

---

# Normalization

Делается через Strategy Pattern.

```python id="6jgjch"
Normalizer

    StandardScaler

    MinMaxScaler

    RobustScaler

    Identity
```

Например

```python id="f68mgp"
class StandardNormalizer:

    def fit(...)

    def transform(...)
```

---

# Dataset

Отдельный Dataset.

```python id="ax2jcd"
class HiggsDataset(Dataset):

    def __getitem__(self,index):

        return

            x

            y
```

Позже сюда будут добавлены

```text id="f0nn4v"
hand-crafted augmentation

Atlas augmentation

TDA augmentation
```

---

# Split Strategy

Очень важно.

Никогда не делать

```python id="zcln5l"
train_test_split()
```

внутри обучения.

Лучше

```text id="q7m7bq"
split.py

↓

SplitGenerator

↓

seed

↓

indices

↓

save indices
```

Получаются

```text id="8ij6lp"
train_idx.npy

val_idx.npy

test_idx.npy
```

Это обеспечивает воспроизводимость.

---

# Statistics Module

Автоматически считает

```text id="a7aw4o"
mean

std

min

max

correlation

class balance
```

и сохраняет

```text id="ihabcs"
statistics.json
```

---

# Cache

После первого запуска

```text id="xndpcl"
CSV

↓

Parquet

↓

Scalers

↓

Statistics

↓

cached
```

Все последующие эксперименты используют

Parquet.

Это существенно ускоряет запуск.

---

# Lightning DataModule

Получается

```python id="7c0k0z"
class HiggsDataModule(

    BaseDataModule

):

    prepare_data()

    setup()

    train_dataloader()

    val_dataloader()

    test_dataloader()
```

---

# DataLoader

Я бы сразу использовал

```python id="x3nqkx"
DataLoader(

    pin_memory=True,

    persistent_workers=True,

    prefetch_factor=4,

    shuffle=True
)
```

для ускорения.

---

# Что будет логироваться

После

```python id="pnjlwm"
prepare_data()
```

автоматически появляются

```text id="m28scd"
dataset size

train size

validation size

test size

feature mean

feature std

label ratio
```

и отправляются в

Weights & Biases.

---

# Unit Tests

Добавляются тесты:

```text id="frcicg"
test_download.py

test_split.py

test_preprocessing.py

test_dataset.py
```

Проверяется:

* корректность скачивания;
* отсутствие пересечения между train/validation/test;
* совпадение размеров выборок;
* корректность нормализации;
* соответствие ожидаемой размерности признаков.

---

# Что получится после этапа 2

На выходе будет полноценный модуль данных, который:

* автоматически загружает и валидирует датасет HIGGS;
* выполняет воспроизводимую предобработку с кэшированием результатов;
* использует сохранённые индексы разбиения для идентичности всех экспериментов;
* предоставляет стандартный `LightningDataModule`;
* легко расширяется на другие HEP-датасеты за счёт общей базовой архитектуры.

После этого можно переходить к **этапу 3 — Model Zoo**, где будут реализованы Encoder, Classifier, Fusion и базовая архитектура для последующего подключения модулей симметрий и TDA.


# equivariant neural network + differentiable TDA (real SOTA version)

Это уже уровень полноценной исследовательской работы. Важно отметить, что **не существует одной общепринятой "SOTA-архитектуры"**, которая одновременно объединяет:

* equivariant neural networks;
* differentiable persistent homology;
* learned symmetry discovery;
* табличные данные HIGGS.

Поэтому такую систему придется собирать из нескольких современных направлений. Ниже — архитектура, которую я бы считал наиболее перспективной для статьи уровня NeurIPS.

---

# Общая архитектура

```text id="3g44wf"
                     Raw HIGGS Event
                            │
                            ▼
              Physics Feature Embedding
                            │
        ┌───────────────────┼────────────────────┐
        │                   │                    │
        ▼                   ▼                    ▼
 Equivariant Encoder   Atlas Symmetry     TDA Branch
        │             Discovery Module          │
        │                   │                   │
        └──────────────┬────┴───────────────┐
                       ▼                    ▼
              Shared Latent Manifold     Persistence Images
                       │                    │
                       └────────┬───────────┘
                                ▼
                      Multi-view Contrastive
                                │
                                ▼
                       Classification Head
```

---

# Часть 1. Equivariant Network

Здесь есть важный момент.

**HIGGS — табличный датасет.**

Поэтому напрямую использовать архитектуры, разработанные для молекул или 3D-точек (например, E(3)-эквивариантные сети), нельзя без преобразования данных.

Есть два реалистичных варианта.

### Вариант A (наиболее практичный)

Построить граф взаимодействий между физическими объектами события (лептоны, джеты, missing ET и т.п.) и использовать **эквивариантную графовую нейронную сеть**.

Пайплайн:

```text id="mrtlfn"
Event

↓

Physics Objects

↓

Graph

↓

Equivariant Message Passing

↓

Latent
```

### Вариант B

Работать в пространстве признаков и явно задавать действие группы:

[
f(gx)=\rho(g)f(x),
]

где (\rho(g)) — представление группы в латентном пространстве.

Этот путь проще реализовать на HIGGS и лучше контролируется экспериментально.

---

# Часть 2. Дифференцируемая TDA

Обычный пайплайн выглядит так:

```text id="rqjuhg"
Embedding

↓

Ripser

↓

Persistence Diagram

↓

detach()

↓

MLP
```

Проблема — потеря градиентов.

В исследовательской версии лучше использовать дифференцируемый слой:

```text id="v2vpyq"
Embedding

↓

Differentiable Persistence Layer

↓

Persistence Image

↓

CNN

↓

Topological Embedding
```

Тогда классификационная ошибка влияет и на топологическое представление.

---

# Часть 3. Persistence Images

Вместо двух-трёх чисел (например, среднего времени жизни классов) строится полноценное изображение.

```text id="5mnj5w"
Persistence Diagram

↓

Gaussian Kernel

↓

2D Grid

↓

Persistence Image
```

Далее это изображение обрабатывается небольшой CNN.

---

# Часть 4. Atlas Symmetry Learning

Вместо MLP, генерирующей преобразование, вводится модель на основе алгебры Ли.

Латентный вектор:

[
z=f_\theta(x)
]

Параметризуем генератор

[
A=\sum_i\alpha_i B_i,
]

где (B_i) — базисные генераторы, а коэффициенты (\alpha_i) обучаются.

Элемент группы получается через

[
g=\exp(A).
]

Если сделать (\alpha_i) случайными величинами, получаем стохастический генератор преобразований.

---

# Часть 5. Multi-view Contrastive Learning

Из одного события формируются несколько представлений:

```text id="gr0oyv"
View 1

Original
```

```text id="obsmu6"
View 2

Hand-crafted symmetry
```

```text id="jlwmkf"
View 3

Atlas symmetry
```

```text id="8opshw"
View 4

Topological embedding
```

Все они проходят через общий энкодер.

Далее используется контрастивная функция потерь (например, InfoNCE или её современные варианты), чтобы представления одного события были близки друг к другу, а разных событий — разделялись.

---

# Часть 6. Совместная функция потерь

Можно использовать следующую композицию:

[
\mathcal L=
\lambda_{cls}\mathcal L_{cls}
+\lambda_{eq}\mathcal L_{equiv}
+\lambda_{sym}\mathcal L_{sym}
+\lambda_{ctr}\mathcal L_{InfoNCE}
+\lambda_{tda}\mathcal L_{tda}
+\lambda_{reg}\mathcal L_{reg}.
]

Это позволяет проводить абляции по каждому компоненту.

---

# Часть 7. Рекомендуемые библиотеки

Для реализации такого проекта я бы ориентировался на следующий стек:

| Задача               | Библиотеки                                                                                                                                                   |
| -------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| Equivariant NN       | `e3nn` (если данные преобразуются в подходящее представление), либо собственная реализация с явным действием группы для табличных признаков                  |
| Graph Processing     | `torch_geometric`                                                                                                                                            |
| Differentiable TDA   | `torch_topological` или другие библиотеки с поддержкой автодифференцирования для топологических слоёв; при необходимости — собственный дифференцируемый слой |
| Persistent Homology  | `GUDHI`, `Ripser.py`                                                                                                                                         |
| Persistence Images   | `persim`                                                                                                                                                     |
| Contrastive Learning | собственная реализация на PyTorch или адаптация из открытых библиотек                                                                                        |

---

# Почему это можно считать архитектурой уровня NeurIPS

Наиболее интересная исследовательская идея здесь — **совместное обучение**:

* эквивариантного представления данных;
* параметризованной группы преобразований (через генераторы алгебры Ли);
* топологических характеристик латентного пространства;
* контрастивной структуры между несколькими представлениями одного события.

При этом стоит помнить, что некоторые элементы (например, применение `e3nn` к HIGGS) требуют содержательной адаптации данных и физической мотивации. Для табличного HIGGS зачастую более обоснованным будет вариант с **явно заданным действием группы в пространстве признаков** или с **графовым представлением физических объектов**, а не прямое применение геометрических архитектур, разработанных для 3D-координат. Это важно будет аргументировать в разделе *Method* и проверить в абляционных экспериментах.
